# Tool Binding in LangChain

**Tool Binding** is the step where you register tools with a Language Model (LLM) so that:

1. The LLM knows what tools are available  
2. It knows what each tool does (via description)  
3. It knows what input format to use (via schema)

---

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import requests
load_dotenv()

True

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")

- Tool Creation -> Tool Binding -> Tool Calling -> Tool Execution

In [6]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a*b

In [10]:
response = llm.invoke("What is 987654 * 123456?")
print(response.content)

987654 multiplied by 123456 equals 121932768.


In [15]:
# Bind tools to LLM
llm_with_tools = llm.bind_tools([multiply])

response = llm_with_tools.invoke("What is 5 multiplied by 6?")
print(response.tool_calls)
# The LLM only generates a tool call request when tools are bound, but it does not automatically execute the tool unless an agent loop runs it.

[{'name': 'multiply', 'args': {'a': 5, 'b': 6}, 'id': 'call_C2e94vzLk9trmp2X9ME3Cjkr', 'type': 'tool_call'}]


# Tool Calling in LangChain

**Tool Calling** is the process where the LLM decides during a conversation that it needs to use a specific tool (function).

When this happens, the LLM generates a structured output containing:

- The name of the tool  
- The arguments to call it with  

⚠️ The LLM does NOT execute the tool.  
It only suggests which tool to use and what inputs to pass.  

The actual execution is handled by:
- LangChain (via an agent loop), or  
- You (manual execution)

---

In [18]:
print(response.tool_calls[0])

{'name': 'multiply', 'args': {'a': 5, 'b': 6}, 'id': 'call_C2e94vzLk9trmp2X9ME3Cjkr', 'type': 'tool_call'}


# Tool Execution in LangChain

**Tool Execution** is the step where the actual Python function (tool) is run using the input arguments that the LLM suggested during tool calling.

---

## In Simpler Words

🧠 The LLM says:

> "Call the `multiply` tool with a=8 and b=7."

⚙️ Tool Execution is when you or LangChain actually run:

```python
multiply(a=8, b=7)

In [20]:
response.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 5, 'b': 6},
 'id': 'call_C2e94vzLk9trmp2X9ME3Cjkr',
 'type': 'tool_call'}

In [21]:
multiply.invoke(response.tool_calls[0])

ToolMessage(content='30', name='multiply', tool_call_id='call_C2e94vzLk9trmp2X9ME3Cjkr')

In [29]:
from langchain_core.messages import ToolMessage
# 1️⃣ Create LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# 2️⃣ Create Tool
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

# 3️⃣ Bind Tool
llm_with_tools = llm.bind_tools([multiply])

# 4️⃣ First Call (Tool Calling)
response = llm_with_tools.invoke("What is 8 multiplied by 7?")

# 5️⃣ Execute Tool
tool_call = response.tool_calls[0]
result = multiply.invoke(tool_call["args"])

# 6️⃣ Send Tool Result Back to Model
tool_message = ToolMessage(
    content=str(result),
    tool_call_id=tool_call["id"]
)

final_response = llm_with_tools.invoke([
    response,
    tool_message
])

print(final_response.content)

The result of multiplying 8 by 7 is 56.


# Currency Converter Tool

In [52]:
EXCHANGE_RATES = {
    ("USD", "INR"): 83,
    ("INR", "USD"): 0.012,
    ("USD", "EUR"): 0.92,
    ("EUR", "USD"): 1.08,
}

@tool
def convert(base_currency: int, conversion_factor: float) -> float:
    """
    Converts a base currency amount into another currency using the given conversion factor.
    
    Args:
        base_currency: The original currency amount to be converted.
        conversion_factor: The exchange rate used for conversion.
        
    Returns:
        The converted currency amount.
    """
    return base_currency * conversion_factor

@tool
def get_conversion_factor(from_currency: str, to_currency: str) -> dict:
    """
    Returns the exchange rate (conversion factor) between two currencies.

    Args:
        from_currency: The source currency code (e.g., USD).
        to_currency: The target currency code (e.g., INR).

    Returns:
        A dictionary containing:
        - from_currency
        - to_currency
        - conversion_rate

        Returns an error message if the rate is unavailable.
    """

    rate = EXCHANGE_RATES.get(
        (from_currency.upper(), to_currency.upper())
    )

    if rate is None:
        return {
            "error": "Conversion rate not available."
        }

    return {
        "from_currency": from_currency.upper(),
        "to_currency": to_currency.upper(),
        "conversion_rate": rate
    }

In [34]:
llm = ChatOpenAI(model="gpt-4o-mini")

llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [40]:
query = 'What is the conversion rate of Dollar to Rupees and how much is 20$ in rupees?'

In [37]:
# Need of Injected Tool Arg
messages = [HumanMessage(query)]
ai_message = llm_with_tools.invoke(messages)
print(ai_message.tool_calls)

[{'name': 'get_conversion_factor', 'args': {'from_currency': 'USD', 'to_currency': 'INR'}, 'id': 'call_i2FkDhoYscxXIbcprcpoRIOV', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency': 20, 'conversion_factor': 0}, 'id': 'call_atJVbFhzTZDa9POp7MwCpRNG', 'type': 'tool_call'}]


In [61]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated
# ❌ “LLM, do NOT try to fill this argument.”  
# ✅ “I (the developer/runtime) will inject this value after running earlier tools.”
@tool
def convert(base_currency: int, conversion_factor: Annotated[float, InjectedToolArg]) -> float:
    """
    Converts a base currency amount into another currency using the given conversion factor.
    
    Args:
        base_currency: The original currency amount to be converted.
        conversion_factor: The exchange rate used for conversion.
        
    Returns:
        The converted currency amount.
    """
    return base_currency * conversion_factor

@tool
def get_conversion_factor(from_currency: str, to_currency: str) -> dict:
    """
    Returns the exchange rate (conversion factor) between two currencies.

    Args:
        from_currency: The source currency code (e.g., USD).
        to_currency: The target currency code (e.g., INR).

    Returns:
        A dictionary containing:
        - from_currency
        - to_currency
        - conversion_rate

        Returns an error message if the rate is unavailable.
    """

    rate = EXCHANGE_RATES.get(
        (from_currency.upper(), to_currency.upper())
    )

    if rate is None:
        return {
            "error": "Conversion rate not available."
        }

    return {
        "from_currency": from_currency.upper(),
        "to_currency": to_currency.upper(),
        "conversion_rate": rate
    }

In [79]:
# Bind Again
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])
                                 
messages = [HumanMessage(query)]
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)
print(ai_message.tool_calls)

[{'name': 'get_conversion_factor', 'args': {'from_currency': 'USD', 'to_currency': 'INR'}, 'id': 'call_lAyKPknej5UavJfCuzWfRuaI', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency': 20}, 'id': 'call_dfmzJz3j2rpMPqLQxBtV97Ia', 'type': 'tool_call'}]


In [80]:
import json
for tool_call in ai_message.tool_calls:
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        conversion_factor = json.loads(tool_message1.content)['conversion_rate']
        messages.append(tool_message1)
    if tool_call['name'] == 'convert':
        # Fetch the args for this tool_call
        tool_call['args']['conversion_factor'] = conversion_factor
        print(tool_call['args'])
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

{'base_currency': 20, 'conversion_factor': 83}


In [81]:
print(messages)

[HumanMessage(content='What is the conversion rate of Dollar to Rupees and how much is 20$ in rupees?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 196, 'total_tokens': 248, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-DBgomPj7NuKw4ZG44pgruBBWwrVko', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c8045-a14e-7752-9c5f-aeb8506abf8d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'from_currency': 'USD', 'to_currency': 'INR'}, 'id': 'call_lAyKPknej5UavJfCuzWfRuaI', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_

In [83]:
response = llm_with_tools.invoke(messages)
print(response.content)

The conversion rate from Dollar (USD) to Rupees (INR) is 83. Therefore, 20 USD is equivalent to 1660 INR.


# This is not agent
